# Logistic Regression, end to end — insurance claim data

This notebook is the **workflow of a data scientist**, not just "call `.fit()`". Every step below is something you'd actually do on a real dataset, in order, with the *reasoning* for each step written out — not just the code.

**Dataset:** `assets/claim.csv` — an insurance bodily-injury claims dataset. Each row is one claim.

**The business question we're answering:** *given what we know about a claim (claimant's age, sex, whether they had insurance, whether they wore a seatbelt, and the claim's loss amount), can we predict whether the claimant will retain an attorney?*

Why this matters in the real world: claims where the claimant hires an attorney tend to be costlier and more contested for the insurer. If an insurer can predict *early* which claims are likely to involve an attorney, they can route those claims to more experienced adjusters, set aside more reserve money, or handle them differently from the start.

This directly follows the Chapter 1 "how to approach a business problem" checklist and Chapter 3's Logistic Regression theory — read those first if you haven't. Refer back to `03-Logistic Regression/theory.md` for the math (Sigmoid, Log Loss, Gradient Descent) behind what `sklearn` does automatically in Step 5 below.

## The data science workflow, at a glance

1. **Load & inspect** — what do we actually have?
2. **EDA (Exploratory Data Analysis)** — understand the data before touching a model
3. **Clean the data** — handle missing values, decide what to do with them
4. **Prepare features & target, split train/test**
5. **Train the Logistic Regression model**
6. **Evaluate** — is it actually any good? (compared to what?)
7. **Interpret** — what did the model actually learn, in plain English?

Every step matters — skipping straight to Step 5 is the #1 beginner mistake (recall Chapter 1's recap: bad problem framing / bad data handling kills more ML projects than bad algorithms).

---
## Step 1 — Load & inspect the data

First question, always: *what do I actually have?* Shape, column names, data types, and a first look at raw rows.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../assets/claim.csv")

print(df.shape)          # (rows, columns) -> (1340, 7)
df.head()                # first 5 rows, to eyeball what the data actually looks like


In [ ]:
df.info()   # column names, dtypes, and non-null counts all in one place


**Reading `df.info()`:** 7 columns, 1340 rows.

| Column | What it is |
|---|---|
| `CASENUM` | A case/claim ID — not a real feature (recall Chapter 1's "does arithmetic on this number mean anything?" nominal-data trap: an ID is nominal, not something to model on) |
| `ATTORNEY` | 1 = claimant retained an attorney, 0 = did not — **this is our target** |
| `CLMSEX` | Claimant's sex (1/0) |
| `CLMINSUR` | Whether the claimant's own car was insured (1/0) |
| `SEATBELT` | Whether the claimant was wearing a seatbelt (1/0) |
| `CLMAGE` | Claimant's age |
| `LOSS` | The claim's total economic loss, in $1000s |

Also notice: `CLMSEX`, `CLMINSUR`, `SEATBELT`, `CLMAGE` are `float64`, not `int64`, even though they look like whole numbers/flags. That's `pandas`' way of telling you **these columns have missing values** — a column with even a single `NaN` can't stay an integer type, so pandas silently upgrades it to float. That's your first real clue, before even running `.isnull()`, that this data isn't clean yet.

---
## Step 2 — EDA (Exploratory Data Analysis)

Before cleaning or modeling anything, understand the data on its own terms. Two things matter most for a classification problem: **the target's distribution** (is it balanced?) and **how each feature relates to the target** (is there even a signal to learn from?).

In [ ]:
df["ATTORNEY"].value_counts()
# 0    685
# 1    655
# -> fairly balanced (not 99/1 like fraud detection) -- good, a lopsided target
#    would need extra handling (e.g. class weighting) we don't need to worry about here


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.countplot(data=df, x="ATTORNEY", ax=axes[0])
axes[0].set_title("Target: did the claimant retain an attorney?")

sns.histplot(df["CLMAGE"].dropna(), bins=30, ax=axes[1])
axes[1].set_title("Distribution of claimant age (CLMAGE)")

plt.tight_layout()
plt.show()


In [ ]:
df.describe()
# quick sanity check on the numeric columns -- CLMAGE ranges 0-95 (0 = infants in
# accidents, plausible for this kind of dataset), LOSS ranges from ~0 to ~174 ($1000s),
# heavily right-skewed (mean 3.8 vs median ~1.07 -- recall statistics-self-learning's
# mean-vs-median gap = a skewness/outlier signal, before even plotting anything)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.boxplot(data=df, x="ATTORNEY", y="CLMAGE", ax=axes[0])
axes[0].set_title("Age vs. Attorney (any visible difference?)")

sns.boxplot(data=df, x="ATTORNEY", y="LOSS", ax=axes[1])
axes[1].set_ylim(0, 20)   # zoom in -- LOSS has extreme outliers that would flatten the plot otherwise
axes[1].set_title("Loss amount vs. Attorney (zoomed in)")

plt.tight_layout()
plt.show()


**Reading these plots:** if a feature's distribution looks *identical* regardless of `ATTORNEY`'s value, that feature probably carries little predictive signal on its own. If the boxes are shifted noticeably, that's a hint the feature matters. (Real interpretation happens after training too, in Step 7 — this is just the "does this look promising at all" pass, before spending any modeling effort.)

---
## Step 3 — Clean the data

Recall from Step 1: `CLMSEX`, `CLMINSUR`, `SEATBELT`, `CLMAGE` all have missing values. A model can't train on `NaN` — every missing value needs a deliberate decision, not a silent default.

In [ ]:
df.isnull().sum()
# CASENUM       0
# ATTORNEY      0
# CLMSEX       12
# CLMINSUR     41
# SEATBELT     48
# CLMAGE      189
# LOSS          0


**The decision to make:** drop rows with missing values, or fill them in (imputation)?

- `CLMAGE` alone is missing 189 out of 1340 rows (~14%) — the biggest gap.
- Combined across all 4 columns, up to 244 rows have *at least one* missing value (some rows overlap on multiple missing columns).

For this notebook we'll **drop rows with any missing value** — the simplest, most transparent choice, and we still keep the large majority of the data (1096 of 1340 rows, ~82%). This is a reasonable first-pass decision; a more advanced pass could impute `CLMAGE` with the median (recall `statistics-self-learning`: median over mean, since `CLMAGE` isn't symmetric) instead of dropping those rows outright — worth trying later once this baseline is working.

In [ ]:
df_clean = df.drop(columns=["CASENUM"])   # CASENUM is an ID, not a feature -- drop it now, not just ignore it later
df_clean = df_clean.dropna()               # drop any row missing ANY value in the remaining columns

print(f"Rows before: {len(df)}, rows after dropping missing values: {len(df_clean)}")
# Rows before: 1340, rows after dropping missing values: 1096


---
## Step 4 — Prepare features & target, then split train/test

Separate the **target** (`ATTORNEY`, what we're predicting) from the **features** (everything else, what we're predicting *from*). Then split into a training set (the model learns from this) and a test set (held back, never seen during training — used only to check the model actually generalizes, not just memorized). This is the exact workflow Chapter 1, Section 2 described in the abstract — here it's real code.

In [ ]:
from sklearn.model_selection import train_test_split

X = df_clean.drop(columns=["ATTORNEY"])   # features: CLMSEX, CLMINSUR, SEATBELT, CLMAGE, LOSS
y = df_clean["ATTORNEY"]                   # target: 0 or 1

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # hold back 20% purely for evaluation
    random_state=42,     # fixed seed -> same split every time we rerun this cell (reproducibility)
    stratify=y,           # keep the 0/1 ATTORNEY ratio roughly the same in both train and test sets
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
# Train: (876, 5), Test: (220, 5)


**Why `stratify=y`?** Recall Step 2 — `ATTORNEY` is roughly 51%/49%. Without `stratify`, a random split *could* accidentally put more 1s in the training set and more 0s in the test set (or vice versa) purely by chance, especially in smaller datasets. `stratify=y` forces both the train and test sets to keep that same ~51/49 ratio, so we're evaluating the model fairly.

---
## Step 5 — Train the Logistic Regression model

This is where `03-Logistic Regression/theory.md` becomes real code. `sklearn`'s `LogisticRegression` does exactly what the theory describes internally: computes `z = m1*x1 + m2*x2 + ... + b`, squashes it through Sigmoid to get a probability, and uses Gradient Descent (theory.md Section 4) to find the `m`'s and `b` that minimize Log Loss (theory.md Section 3) across the training data.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)   # max_iter: how many gradient descent steps to allow before giving up
model.fit(X_train, y_train)                  # this is the actual "training" -- finds the best m's and b


In [ ]:
# The model's learned parameters -- these ARE the "2 numbers" idea from Chapter 1's
# Parametric section, just one m per feature instead of one, plus a single b.
for feature, m in zip(X.columns, model.coef_[0]):
    print(f"{feature:>10}: m = {m:+.4f}")
print(f"{'intercept':>10}: b = {model.intercept_[0]:+.4f}")
# CLMSEX: m = +0.4193
# CLMINSUR: m = +0.7992
# SEATBELT: m = -0.4963
# CLMAGE: m = +0.0055
# LOSS: m = -0.3881
# intercept: b = -0.3281


---
## Step 6 — Evaluate: is this model actually any good?

A number like "67% accuracy" means nothing on its own — good compared to *what*? Always compare against a **baseline**: the simplest possible non-model prediction (recall Chapter 1's recap, Step 6: "start simple" — the baseline itself is the simplest possible starting point).

In [ ]:
# Baseline: what if we just always predicted the MAJORITY class, ignoring every feature?
majority_class = y_train.mode()[0]
baseline_accuracy = (y_test == majority_class).mean()

print(f"Majority class in training data: {majority_class}")
print(f"Baseline accuracy (always predict {majority_class}): {baseline_accuracy:.3f}")
# Majority class in training data: 0
# Baseline accuracy (always predict 0): 0.527


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

y_pred = model.predict(X_test)              # the final 0/1 prediction (threshold at 0.5, theory.md Section 2 Step 3)
y_proba = model.predict_proba(X_test)[:, 1]  # the raw probability BEFORE thresholding (theory.md Section 2 Step 2)

accuracy = accuracy_score(y_test, y_pred)
print(f"Model accuracy: {accuracy:.3f}")
print(f"Baseline accuracy: {baseline_accuracy:.3f}")
print(f"Improvement over baseline: {accuracy - baseline_accuracy:+.3f}")
# Model accuracy: 0.673
# Baseline accuracy: 0.527
# Improvement over baseline: +0.145


**Reading this:** the model beats the "always guess the majority class" baseline by about 14.5 percentage points (67.3% vs 52.7%). That confirms the features actually carry *some* real signal about attorney involvement — not a huge edge, but a genuine one, on a hard, noisy real-world question. (If the model had landed at ~53%, barely beating the baseline, that would be a sign the features aren't very predictive for this target, and no amount of model tuning would fix that — recall Chapter 1: bad data/features, not the algorithm, is usually the real bottleneck.)

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted: No Attorney", "Predicted: Attorney"],
            yticklabels=["Actual: No Attorney", "Actual: Attorney"])
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


**Reading a confusion matrix:**
- **Top-left (True Negatives):** correctly predicted "No Attorney"
- **Bottom-right (True Positives):** correctly predicted "Attorney"
- **Top-right (False Positives):** predicted "Attorney" but actually wasn't — a false alarm
- **Bottom-left (False Negatives):** predicted "No Attorney" but actually was — a missed case

Which mistake is worse depends on the business (recall Chapter 1's recap, Step 5: "is a wrong prediction cheap or costly?"). Here, missing a case that *will* involve an attorney (a False Negative) probably costs the insurer more — they'd be caught under-prepared for a contested claim — than a False Positive (over-preparing for a claim that turns out simple).

In [ ]:
print(classification_report(y_test, y_pred, target_names=["No Attorney", "Attorney"]))


**Precision vs. Recall, quickly:**
- **Precision** (for class "Attorney"): of everything the model *predicted* as "Attorney," what fraction actually was? Answers "when the model raises a flag, can I trust it?"
- **Recall** (for class "Attorney"): of everything that *actually was* "Attorney," what fraction did the model catch? Answers "how many real cases does the model actually find?"

There's usually a tradeoff between the two — covered in depth in a later evaluation-metrics chapter. For now, just know **accuracy alone is never the full picture** — always check precision/recall too, especially when the cost of the two mistake types (Step 6's False Positive vs. False Negative) isn't equal.

In [ ]:
auc = roc_auc_score(y_test, y_proba)
print(f"ROC-AUC: {auc:.3f}")
# ROC-AUC: 0.745
# Rough reading: 0.5 = no better than random guessing, 1.0 = perfect separation.
# 0.745 means the model has a genuinely useful, if imperfect, ability to rank
# "more likely to hire an attorney" claims above "less likely" ones.


---
## Step 7 — Interpret: what did the model actually learn?

This is the step that makes Logistic Regression genuinely valuable to a business, beyond just its predictions (recall Chapter 1, Section 5: parametric models are **interpretable** — you can read the formula and explain *why*, unlike a black-box model). Go back to the coefficients from Step 5 and read them in plain English.

In [ ]:
coef_table = pd.DataFrame({
    "feature": X.columns,
    "coefficient (m)": model.coef_[0],
}).sort_values("coefficient (m)", key=abs, ascending=False)

coef_table


**Reading each coefficient** (recall `03-Logistic Regression/theory.md` Section 2 — each `m` says how `z` moves, before Sigmoid squashes it into a probability; a **positive** `m` pushes the probability of "Attorney" *up*, a **negative** `m` pushes it *down*):

- **`CLMINSUR` (+0.80, the strongest effect):** claimants who *had* insurance were **more** likely to retain an attorney. A plausible read: an insured claimant may feel more confident pursuing a claim seriously (with legal backing), or insurance involvement itself makes claims more likely to become contested.
- **`SEATBELT` (-0.50):** wearing a seatbelt is associated with a **lower** likelihood of attorney involvement — plausibly because seatbelt-wearers tend to have less severe injuries, so there's less to dispute or litigate.
- **`CLMSEX` (+0.42):** one sex codes as more likely to retain an attorney than the other, holding other factors constant.
- **`LOSS` (-0.39):** counter-intuitive at first glance — *higher* loss amounts are associated with a **lower** predicted probability of attorney involvement, when the other features are held constant. This is exactly why Chapter 2/3 stressed "holding all other features constant" — this is a raw statistical association within *this* dataset and model, not a claim about causation; a real analysis would dig further here (e.g. check for interaction effects) before trusting this number as a business insight.
- **`CLMAGE` (+0.0055, tiny):** age has a very small effect — each additional year barely moves the prediction. Look at the *scale* of `CLMAGE` values (0-95) vs. the other 0/1 features before concluding age "doesn't matter" — a small coefficient on a wide-ranging variable can still matter in aggregate. Worth normalizing features and re-checking in a later pass.

**The important caveat, stated plainly:** these are *associations the model found in this specific dataset*, not proven causes. "Insurance is associated with higher attorney involvement" is not the same claim as "having insurance causes people to hire attorneys." Recall Chapter 1's "human interpretation" step for unsupervised clustering — the same caution applies here: the model finds a pattern; deciding *why* that pattern exists, and whether it's safe to act on, is a human/business judgment call, not something the model itself proves.

---
## Recap — what we actually did, and why, in order

1. **Loaded & inspected** — found the target (`ATTORNEY`), noticed missing values *before* even running `.isnull()`, just from dtypes.
2. **EDA** — checked the target was reasonably balanced (no special class-imbalance handling needed), eyeballed whether features looked different across the two classes.
3. **Cleaned** — dropped the ID column and rows with missing values, with a stated, honest tradeoff (simplicity vs. keeping every row) rather than silently doing something automatic.
4. **Split train/test** — with `stratify` to keep the target's ratio consistent, so evaluation would be fair.
5. **Trained** — let `sklearn` run the exact Sigmoid + Log Loss + Gradient Descent process from `theory.md`.
6. **Evaluated against a baseline** — not just "accuracy = 0.67" in isolation, but "0.67 vs. a 0.53 baseline," plus confusion matrix, precision/recall, and ROC-AUC, because accuracy alone hides how the two types of mistakes are distributed.
7. **Interpreted** — read the actual learned coefficients back into plain-English business language, and were explicit about the association-vs-causation line.

This 7-step shape — not just step 5 — is what "doing Logistic Regression as a data scientist" actually means in practice. Steps 1-4 and 6-7 are usually the majority of the real work; the `model.fit()` call in Step 5 is often the smallest part.